# Runoff contribution


In [ ]:
# general
import geopandas as gpd
import xarray as xr

# plotting
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go

from src.utils import load_config
cfg = load_config()

## Data

In [ ]:
common_period = slice("2001-01-01", "2024-12-31")

# polygons / shapes
hydroatlas_shape = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")
hydroatlas_shape['basin_cenlat_rounded'] = hydroatlas_shape['basin_cenlat'].round(0)
hydroatlas_shape = hydroatlas_shape.sort_values('aridity_ERA5')

# total runoff in mm/day
hydroatlas_runoff = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc", chunks = "auto").q_mm_sim
hydroatlas_runoff = hydroatlas_runoff.sel(date=common_period)

# glacier melt in m3/s -> mm/day
hydroatlas_gmelt = xr.open_dataset("data/glacier/run_output_historical.nc", chunks = "auto").glacier_melt
hydroatlas_gmelt = hydroatlas_gmelt.reindex(gauge_id=hydroatlas_shape.index, fill_value=0)
hydroatlas_gmelt = hydroatlas_gmelt.sel(date=common_period)
hydroatlas_gmelt = (hydroatlas_gmelt * 1000 * 86400) / (hydroatlas_shape.basin_area.values * 1e6)[:, None]

# annual ratio 
ratio = xr.merge([hydroatlas_runoff, hydroatlas_gmelt], join="outer")
ratio = ratio.resample(date="YS").sum().mean(dim = "date")
ratio = (ratio.glacier_melt * 100 / ratio.q_mm_sim).to_series()
hydroatlas_shape = hydroatlas_shape.join(ratio.rename("runoff_gmelt_ratio"))


## Plot

In [ ]:
colorscale = ["#f3e79b", "#eb7f86", "#ce6693", "#a059a0", "#5c53a5"]
scaler = MinMaxScaler(feature_range=(10, 35))
slides = False

fig = go.Figure()

# Plot Nº3: Glacier melt to runoff ratio (+ aridity)
fig.add_trace(
    go.Scatter(
        x=hydroatlas_shape["glacier_area_RGI60"],
        y=hydroatlas_shape["runoff_gmelt_ratio"],
        mode="markers",
        marker=dict(
            size=scaler.fit_transform(hydroatlas_shape[["basin_area"]]).flatten(),
            color= '#31688e' if slides else hydroatlas_shape.aridity_ERA5,
            colorscale=colorscale,
            opacity=0.9,
            colorbar=dict(
                title="Aridity<br>index<br>(PET/P)<br> ",
                len=0.99,
                thickness=20,
                ticks="outside",
                tickvals=[0, 0.2, 0.4, 0.6, 0.8, 1],
                ticktext=["0", "0.2", "0.4", "0.6", "0.8", ">1"],
            ),
            line=dict(width=0.5, color="#9a9a9a"),
            cmax=1,
            cmin=0,
            showscale=True,
        ),
        showlegend=False,
    ),
)

fig.add_trace(
    go.Scatter(
        x=[-3, 102],
        y=[-3, 102],
        mode="lines",
        name="1:1 line",
        line=dict(color="grey", width = 1, dash="dot"),
        showlegend=False,
    ),
)

# Legend for point size (basin area)
size_legend_values = [100, 1000, 10000]
size_legend_sizes = scaler.transform([[v] for v in size_legend_values]).flatten()

for value, size in zip(size_legend_values, size_legend_sizes):
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(size=size, color="#9a9a9a", line=dict(width=0.5, color="#9a9a9a")),
            legendgroup="size",
            name=f"{value:,.0f}",
            showlegend=True,
        ),
    )

fig.add_annotation(
    xref="paper",
    yref="paper",
    x=0.40,
    y=1.035,
    xanchor="right",
    yanchor="middle",
    text="Basin area (km²):",
    showarrow=False,
    font=dict(size=14, family="Arial"),
)

fig.add_annotation(
    x=70,
    y=30,
    text="Glaciers contribute<br><b>more</b> proportionally",
    showarrow=True,
    arrowhead=5,
    arrowwidth=2,
    arrowsize=1,
    arrowside="start+end",
    ax=-180,
    ay=-180,
    font=dict(size=17, family="Arial"),
)

fig.add_annotation(
    x=75,
    y=23,
    text="Glaciers contribute<br><b>less</b> proportionally",
    showarrow=False,
    font=dict(size=17, family="Arial"),
)

fig.update_yaxes(title="Glacier melt rel. total runoff (%)", title_standoff = 5, range = (-4,102))
fig.update_xaxes(title="Glacier area (%)", title_standoff = 0, range = (-4,102))

# Update axes + annotations
fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False)
fig.update_yaxes(ticks="outside", griddash="dot", zeroline=False)

# Update layout
fig.update_layout(
    autosize=False, 
    barmode = "overlay",
    bargap=0.1,
    legend=dict(orientation="h", y=1.07,  x=0.40, bgcolor="rgba(255,255,255,0)", traceorder="normal", tracegroupgap=0),
    width=600, 
    height=500, 
    boxmode ='group',
    plot_bgcolor="rgba(229,236,246,0.7)",
    margin=dict(l=10, r=10, b=10, t=10), 
    font=dict(size=15, family="Arial"),
)

suffix = "_slides" if slides else ""
fig.write_image(f"figures/fig_runoff_glacier_area{suffix}.svg", width=600, height=500)
fig.show()